Original correct count: 4082  
Removed count: 36 = 1个框外 + 35没有坐标

AOI: 
Identified waste: 
- 4,046 = ZWL + Faith + GSVI  
- 3,897 = GSVI 
- 56 = Faith
- 93 = ZWL

Oringinal Image SVI: 
- 457,111 = ZWL + Faith + GSVI
- 453,364 = GSVI (113,341 - Pans)
- 817 = Faith
- 2,930 = ZWL

Nairobi - SHP: 
- 3,385 = ZWL + Faith + GSVI  
- 3,236 = GSVI
- 56 = Faith
- 93 = ZWL

In [ ]:
import warnings
import numpy as np
import pandas as pd
import geopandas as gpd
from pathlib import Path

import seaborn as sns
import matplotlib as mpl
import matplotlib.pyplot as plt
import matplotlib.cm as cm
import matplotlib.patches as mpatches
import matplotlib.colors as mcolors
from matplotlib import rcParams
import matplotlib.patheffects as path_effects
from matplotlib_scalebar.scalebar import ScaleBar

from shapely.geometry import box, LineString, Point, MultiPoint
from shapely.ops import unary_union, polygonize

from scipy.optimize import curve_fit
from sklearn.cluster import DBSCAN
from sklearn.neighbors import NearestNeighbors
from sklearn.metrics import silhouette_score, r2_score, mean_squared_error

import hdbscan
from collections import Counter

projected_crs = "EPSG:21037"
DATA_DIR = Path('/Users/wenlanzhang/Downloads/PhD_UCL/Data/Waste/')

# Boundary
## Nairobi

In [ ]:
# import geopandas as gpd
# import matplotlib.pyplot as plt

# Read file
boundary = gpd.read_file("/Users/wenlanzhang/Downloads/PhD_UCL/Data/Shp/NariobiShp/Shp_from_Constituency/Nairobi_shp_C.shp")

# Plot
fig, ax = plt.subplots(figsize=(8, 8))
boundary.plot(ax=ax, edgecolor="black", facecolor="none")  # just outlines
plt.title("Nairobi Boundary")
plt.show()

## AOI

In [ ]:
# Read file
AOI_NAI = gpd.read_file("/Users/wenlanzhang/Downloads/PhD_UCL/Data/Waste/Angela/AOI_NAI.gpkg")

# Plot
fig, ax = plt.subplots(figsize=(8, 8))
AOI_NAI.plot(ax=ax, edgecolor="black", facecolor="none")  # just outlines
plt.title("AOI_NAI Boundary")
plt.show()

# Image

## SVI Correct

In [ ]:
Correct_SVI = pd.read_csv('/Users/wenlanzhang/Downloads/PhD_UCL/Data/Waste/img/Correct_SVI.csv')
# Correct_SVI['img_dir'].unique()
# Correct_SVI[(Correct_SVI['img_dir'] != 'Faith/')&(ttt['img_dir'] != 'ZWL/')]
Correct_SVI

In [ ]:
geometry = [Point(xy) for xy in zip(Correct_SVI["lon"], Correct_SVI["lat"])]
Correct_SVI_gdf = gpd.GeoDataFrame(Correct_SVI, geometry=geometry, crs="EPSG:4326")  # WGS84

Correct_SVI_gdf

### Clip to Nairobi

In [ ]:
Correct_SVI_gdf_clipped = gpd.clip(Correct_SVI_gdf, boundary)
Correct_SVI_gdf_clipped[(Correct_SVI_gdf_clipped['img_dir'] != 'Faith/')&(Correct_SVI_gdf_clipped['img_dir'] != 'ZWL/')]

# len(Correct_SVI_gdf_clipped[(Correct_SVI_gdf_clipped['img_dir'] == 'ZWL/')])
# Correct_SVI_gdf_clipped

### Clip to AOI

In [ ]:
Correct_SVI_gdf_clipped = gpd.clip(Correct_SVI_gdf, AOI_NAI)
Correct_SVI_gdf_clipped[(Correct_SVI_gdf_clipped['img_dir'] != 'Faith/')&(Correct_SVI_gdf_clipped['img_dir'] != 'ZWL/')]
# len(Correct_SVI_gdf_clipped[(Correct_SVI_gdf_clipped['img_dir'] == 'Faith/')])
# Correct_SVI_gdf_clipped

##  ALL SVI

In [ ]:
Combined_SVI = pd.read_csv('/Users/wenlanzhang/Downloads/PhD_UCL/Data/Waste/img/Combined_SVI.csv')
# Combined_SVI['img_dir'].unique()
# Combined_SVI[(Combined_SVI['img_dir'] != 'Faith/')&(Combined_SVI['img_dir'] != 'ZWL/')]
# Combined_SVI[Combined_SVI['img_dir'] == 'ZWL/']

geometry = [Point(xy) for xy in zip(Combined_SVI["lon"], Combined_SVI["lat"])]
Combined_SVI_gdf = gpd.GeoDataFrame(Combined_SVI, geometry=geometry, crs="EPSG:4326")  # WGS84
Combined_SVI_gdf

In [ ]:
Combined_GSVI_gdf = Combined_SVI_gdf[(Combined_SVI_gdf['img_dir'] != 'Faith/')&(Combined_SVI['img_dir'] != 'ZWL/')]
Combined_GSVI_gdf

In [ ]:
# Correct_SVI_Nairobi = gpd.clip(Combined_SVI_gdf, boundary)
# Correct_SVI_Nairobi

Correct_GSVI_Nairobi = gpd.clip(Combined_GSVI_gdf, boundary)
Correct_GSVI_Nairobi

In [ ]:
# Correct_SVI_AOI = gpd.clip(Combined_SVI_gdf, AOI_NAI)
# Correct_SVI_AOI

# Correct_GSVI_AOI = gpd.clip(Combined_GSVI_gdf, AOI_NAI)
# Correct_GSVI_AOI

In [ ]:
# AOI: 
# 456208/4 = 114052.0   # All
453160/4 = 113290      # GSVI

In [ ]:
306420/4 = 76605

# 309399 # All

In [ ]:
113341*4